# 🔴 VOIS RAG Hands-On Lab
### Build a RAG Q&A Bot over a Knowledge Base — Step by Step

**What you'll build:** A Q&A system over a fictional VOIS Employee Handbook using:
- 🔪 **LangChain** for chunking & orchestration
- 🗄 **ChromaDB** for vector storage (no API key needed)
- 🤗 **HuggingFace sentence-transformers** for free embeddings
- 🤖 **OpenAI GPT-4o** for generation *(needs your API key)*
- 📊 **RAGAS** for automated evaluation

---
**Cells map to the slides:**
| Cell | Slide topic |
|------|-------------|
| 1 | Setup |
| 2–3 | Fundamentals: chunking |
| 4–5 | Fundamentals: embeddings + retrieval |
| 6 | Generation (RAG answer) |
| 7 | 🎯 Try This: change chunk_size |
| 8 | Advanced: Hybrid Search |
| 9 | Advanced: Reranking |
| 10 | 🎯 Try This: hybrid vs dense |
| 11–12 | Evaluation: RAGAS |

> ⏱ **Target time: 25 minutes** (the last 5 min of the session)

---
## ⚙️ Cell 1 — Install Dependencies
Run this first. Takes ~2 minutes on Colab.

In [ ]:
%%capture
!pip install langchain langchain-community langchain-openai \
             chromadb sentence-transformers \
             rank_bm25 openai \
             ragas datasets

print("✅ All packages installed!")

---
## 📄 Cell 2 — Load the Knowledge Base
We'll use a **fictional VOIS Employee Handbook** as our document corpus.
No file upload needed — it's built into this notebook.

In [ ]:
from langchain.schema import Document

# ── Fictional VOIS Employee Handbook ──────────────────────────────────────
raw_docs = [
    Document(
        page_content="""Annual Leave Policy: VOIS employees are entitled to 21 working days of annual leave per calendar year. 
        Leave must be approved by the direct line manager at least 2 weeks in advance. 
        Unused leave of up to 5 days may be carried over to the following year. 
        Employees who join mid-year receive leave on a pro-rata basis. 
        Annual leave requests should be submitted through the VOIS HR portal (hr.vois.com).""",
        metadata={"source": "HR_Handbook", "section": "Leave Policy"}
    ),
    Document(
        page_content="""Sick Leave Policy: VOIS provides up to 14 days of paid sick leave per year. 
        A medical certificate is required for absences exceeding 3 consecutive days. 
        Sick leave does not accumulate and cannot be converted to cash. 
        In cases of prolonged illness (more than 30 days), employees may apply for extended medical leave 
        subject to approval by HR and the Medical Committee.""",
        metadata={"source": "HR_Handbook", "section": "Leave Policy"}
    ),
    Document(
        page_content="""Remote Work Policy: VOIS employees in eligible roles may work remotely up to 3 days per week. 
        Remote work days must be agreed with the line manager at the start of each month. 
        Employees are expected to be available during core hours (10:00 – 15:00 Cairo time) regardless of location. 
        A stable internet connection and a dedicated workspace are required for remote work. 
        New employees in their first 90 days are required to work on-site full-time.""",
        metadata={"source": "HR_Handbook", "section": "Work Arrangements"}
    ),
    Document(
        page_content="""Performance Review Cycle: VOIS follows a bi-annual performance review cycle. 
        Mid-year reviews take place in June and year-end reviews in December. 
        Performance is rated on a 5-point scale: Exceptional, Exceeds Expectations, Meets Expectations, 
        Needs Improvement, and Unsatisfactory. 
        Ratings directly influence the annual bonus and salary increment decisions. 
        Employees rated 'Needs Improvement' for two consecutive cycles may be placed on a Performance Improvement Plan (PIP).""",
        metadata={"source": "HR_Handbook", "section": "Performance"}
    ),
    Document(
        page_content="""Training and Development: VOIS employees are entitled to a personal training budget of EGP 15,000 per year. 
        This budget can be used for external courses, certifications, or conferences approved by HR. 
        VOIS also provides access to LinkedIn Learning, Coursera for Business, and an internal LMS. 
        Employees seeking professional certifications (e.g., AWS, Azure, PMP) can apply for full reimbursement 
        via the Certification Sponsorship Program, subject to a 1-year stay-back clause.""",
        metadata={"source": "HR_Handbook", "section": "Development"}
    ),
    Document(
        page_content="""IT Equipment Policy: All employees receive a laptop upon joining. 
        The standard configuration is a Dell Latitude or MacBook Pro depending on role requirements. 
        Employees in AI/Data roles may request a GPU-enabled workstation through the IT request portal. 
        Equipment must be returned in good condition upon resignation or termination. 
        Loss or damage due to negligence may result in a financial deduction from the final settlement.""",
        metadata={"source": "IT_Policy", "section": "Equipment"}
    ),
    Document(
        page_content="""Data Security Policy: All VOIS employees must complete the annual Cybersecurity Awareness training. 
        Customer data must never be stored on personal devices or shared via personal email accounts. 
        Access to production systems requires MFA (Multi-Factor Authentication). 
        Any suspected security incident must be reported to the Security Operations Center (SOC) within 1 hour. 
        Violation of the data security policy may result in disciplinary action up to and including termination.""",
        metadata={"source": "IT_Policy", "section": "Security"}
    ),
    Document(
        page_content="""Employee Benefits: VOIS provides comprehensive medical insurance for employees and their immediate family 
        (spouse and up to 3 children). 
        The insurance covers inpatient, outpatient, dental, and optical treatment. 
        Employees also receive life insurance coverage equal to 3 years of their annual gross salary. 
        A subsidised meal allowance of EGP 2,500 per month is provided for on-site days. 
        VOIS also offers a shuttle bus service covering major Cairo and Giza districts.""",
        metadata={"source": "HR_Handbook", "section": "Benefits"}
    ),
    Document(
        page_content="""Promotion and Career Progression: Career progression at VOIS follows a defined career ladder 
        from Junior (L1) to Principal (L5) within each function. 
        Promotions are reviewed annually in the December performance cycle. 
        Eligibility requires a minimum of 12 months in the current role and a performance rating of 
        'Exceeds Expectations' or above in the last review cycle. 
        Promotion decisions are made by the direct manager and approved by the department head.""",
        metadata={"source": "HR_Handbook", "section": "Career"}
    ),
    Document(
        page_content="""Code of Conduct: VOIS employees are expected to act with integrity, respect, and professionalism. 
        Harassment, discrimination, or bullying of any kind will not be tolerated. 
        Employees must declare any conflict of interest, including investments in competitor companies. 
        Accepting gifts from vendors exceeding EGP 500 in value must be approved by the Ethics Committee. 
        Whistleblower protections are in place for employees who report misconduct in good faith.""",
        metadata={"source": "HR_Handbook", "section": "Conduct"}
    ),
]

print(f"✅ Loaded {len(raw_docs)} documents")
print(f"📄 Sample doc preview:\n{raw_docs[0].page_content[:200]}...")

---
## 🔪 Cell 3 — Chunk the Documents
**Slide reference:** *Chunking — The Most Impactful Hyperparameter*

We use `RecursiveCharacterTextSplitter` with the recommended defaults:
- `chunk_size = 512` tokens  
- `chunk_overlap = 50` tokens (≈10%)

🎯 **Try This (Cell 7 below):** Come back and change these numbers after you see the RAGAS scores.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Tune these! ───────────────────────────────────────────────────────────
CHUNK_SIZE    = 512   # try 256 or 1024
CHUNK_OVERLAP = 50    # try 0 or 100
# ─────────────────────────────────────────────────────────────────────────

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],  # tries each in order
)

chunks = splitter.split_documents(raw_docs)

print(f"✅ {len(raw_docs)} documents → {len(chunks)} chunks")
print(f"📦 Avg chunk length: {sum(len(c.page_content) for c in chunks)//len(chunks)} chars")
print(f"\n--- Chunk 0 ---\n{chunks[0].page_content}")
print(f"\n--- Chunk 1 ---\n{chunks[1].page_content}")

---
## 🗄 Cell 4 — Embed & Store in ChromaDB
**Slide reference:** *Embeddings & Vector Stores*

We use `all-MiniLM-L6-v2` from HuggingFace — **free, no API key needed**, runs on Colab CPU in seconds.

> For production on Azure you'd swap this for `text-embedding-3-large` via `AzureOpenAIEmbeddings`.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("⏳ Loading embedding model (first time downloads ~80MB)...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Build ChromaDB vector store in memory
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="vois_handbook",
)

print(f"✅ Embedded and stored {vectorstore._collection.count()} chunks in ChromaDB")

---
## 🔍 Cell 5 — Dense Retrieval
**Slide reference:** *The RAG Pipeline — Retrieval phase*

Query the vector store and see what comes back — before we call the LLM.

In [ ]:
# ── Try different questions! ──────────────────────────────────────────────
QUESTION = "How many days of annual leave do I get?"
# Other questions to try:
# QUESTION = "Can I work from home?"
# QUESTION = "What happens if I lose my laptop?"
# QUESTION = "How do I get promoted?"
# QUESTION = "What is the training budget?"
TOP_K = 3
# ─────────────────────────────────────────────────────────────────────────

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})
retrieved_chunks = retriever.invoke(QUESTION)

print(f"🔍 Query: '{QUESTION}'")
print(f"📦 Retrieved {len(retrieved_chunks)} chunks:\n")
for i, chunk in enumerate(retrieved_chunks):
    print(f"--- Chunk {i+1} (source: {chunk.metadata.get('section', 'N/A')}) ---")
    print(chunk.page_content[:300])
    print()

---
## 🤖 Cell 6 — Generate Answer with GPT-4o
**Slide reference:** *The RAG Pipeline — Generation phase*

Paste your OpenAI API key below. If you don't have one, the cell will show the context without calling the LLM.

In [ ]:
import os

# ── Add your OpenAI API key ───────────────────────────────────────────────
OPENAI_API_KEY = "sk-..."   # ← paste your key here
# ─────────────────────────────────────────────────────────────────────────

# Build the prompt
context = "\n\n".join([f"[Source: {c.metadata.get('section')}]\n{c.page_content}" 
                        for c in retrieved_chunks])

system_prompt = """You are a helpful VOIS HR assistant. 
Answer ONLY based on the provided context. 
If the answer is not in the context, say 'I don't have that information in the handbook.'
Always cite the source section."""

user_prompt = f"""Context:
{context}

Question: {QUESTION}
"""

if OPENAI_API_KEY.startswith("sk-") and len(OPENAI_API_KEY) > 10:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=300,
    )
    answer = response.choices[0].message.content
    print(f"❓ Question: {QUESTION}")
    print(f"\n💬 Answer:\n{answer}")
else:
    print("⚠️  No API key provided — showing prompt that would be sent to GPT-4o:\n")
    print("SYSTEM:", system_prompt)
    print("\nUSER:", user_prompt[:500], "...")

---
## 🎯 Cell 7 — TRY THIS: Tune Chunk Size
**Slide reference:** *Chunking — The Most Impactful Hyperparameter*

Go back to **Cell 3**, change `CHUNK_SIZE` to `256` or `1024`, then **re-run cells 3 → 4 → 5 → 6**.

This cell scores retrieval quality manually so you can compare:

In [ ]:
# Manual retrieval quality check — no API key needed
test_questions = [
    ("How many days of annual leave do I get?", "21 working days"),
    ("Can I work from home every day?",          "up to 3 days per week"),
    ("What is my training budget?",              "EGP 15,000"),
    ("What happens if I get a bad performance review?", "Performance Improvement Plan"),
    ("Is dental covered by insurance?",          "dental"),
]

hits = 0
print(f"Testing with chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}\n")
print(f"{'Question':<50} {'Expected keyword':<30} {'Found?'}")
print("-" * 95)

for q, keyword in test_questions:
    docs = retriever.invoke(q)
    combined = " ".join(d.page_content for d in docs).lower()
    found = keyword.lower() in combined
    hits += found
    print(f"{q[:48]:<50} {keyword:<30} {'✅' if found else '❌'}")

print(f"\n📊 Retrieval Hit Rate: {hits}/{len(test_questions)} = {hits/len(test_questions)*100:.0f}%")
print("\n💡 Try chunk_size=256 and 1024 — which gives the best score?")

---
## 🔀 Cell 8 — Hybrid Search (Dense + BM25)
**Slide reference:** *Hybrid Search + Reranking*

Dense search misses exact keywords. BM25 (sparse) catches them. Combining both = best of both worlds.

**Fusion formula:** `score = α × dense_score + (1−α) × bm25_score`

In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

# ── Tune the fusion weight ────────────────────────────────────────────────
ALPHA = 0.7   # 0 = pure BM25, 1 = pure dense, 0.7 = recommended default
# ─────────────────────────────────────────────────────────────────────────

# Build BM25 index
tokenized_corpus = [c.page_content.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

def hybrid_search(query: str, k: int = 3, alpha: float = ALPHA):
    """Combine dense (Chroma) + sparse (BM25) scores."""
    # Dense scores
    dense_results = vectorstore.similarity_search_with_score(query, k=len(chunks))
    dense_scores = {doc.page_content: 1 - score for doc, score in dense_results}  # convert distance → similarity

    # BM25 scores
    bm25_raw = bm25.get_scores(query.lower().split())
    bm25_max = bm25_raw.max() + 1e-9
    bm25_norm = bm25_raw / bm25_max  # normalise to [0,1]

    # Fuse scores
    fused = []
    for i, chunk in enumerate(chunks):
        d = dense_scores.get(chunk.page_content, 0)
        b = bm25_norm[i]
        combined = alpha * d + (1 - alpha) * b
        fused.append((chunk, combined))

    fused.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in fused[:k]]

# Compare dense vs hybrid on a keyword-heavy query
keyword_query = "EGP 15,000 training certification reimbursement"

dense_results  = retriever.invoke(keyword_query)
hybrid_results = hybrid_search(keyword_query, k=3)

print(f"🔍 Query: '{keyword_query}'\n")
print("📌 Dense only — top result:")
print(dense_results[0].page_content[:200])
print("\n📌 Hybrid (α=0.7) — top result:")
print(hybrid_results[0].page_content[:200])
print("\n💡 For queries with exact numbers/codes, hybrid usually ranks the right chunk higher.")

---
## 🏆 Cell 9 — Reranking with Cross-Encoder
**Slide reference:** *Hybrid Search + Reranking — Pipeline: top-100 cheap → top-5 precise*

A cross-encoder reads query + document **together** for better relevance scoring.

In [ ]:
from sentence_transformers import CrossEncoder

print("⏳ Loading cross-encoder reranker (~100MB)...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ Reranker loaded!")

def rerank(query: str, candidates, top_k: int = 3):
    """Re-score candidates with cross-encoder, return top_k."""
    pairs = [[query, doc.page_content] for doc in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:top_k]], [score for _, score in ranked[:top_k]]

# Retrieve a wider pool first (top-10), then rerank to top-3
broad_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
broad_results   = broad_retriever.invoke(QUESTION)
reranked_docs, rerank_scores = rerank(QUESTION, broad_results, top_k=3)

print(f"\n🔍 Query: '{QUESTION}'")
print(f"📦 Retrieved {len(broad_results)} candidates → reranked to top 3\n")
for i, (doc, score) in enumerate(zip(reranked_docs, rerank_scores)):
    print(f"Rank {i+1}  (score: {score:.3f})  [{doc.metadata.get('section')}]")
    print(doc.page_content[:200])
    print()

---
## 🎯 Cell 10 — TRY THIS: Compare Dense vs Hybrid vs Reranked
**Slide reference:** *Try This on Hybrid slide*

Run the same test suite from Cell 7 — but now with hybrid + reranking:

In [ ]:
print(f"{'Question':<50} {'Keyword':<28} {'Dense':>6} {'Hybrid':>7} {'Rerank':>7}")
print("-" * 105)

dense_hits = hybrid_hits = rerank_hits = 0

for q, keyword in test_questions:
    # Dense
    d_docs  = retriever.invoke(q)
    d_found = keyword.lower() in " ".join(d.page_content for d in d_docs).lower()

    # Hybrid
    h_docs  = hybrid_search(q, k=3)
    h_found = keyword.lower() in " ".join(d.page_content for d in h_docs).lower()

    # Hybrid + Rerank
    broad   = vectorstore.as_retriever(search_kwargs={"k":10}).invoke(q)
    r_docs, _ = rerank(q, broad, top_k=3)
    r_found = keyword.lower() in " ".join(d.page_content for d in r_docs).lower()

    dense_hits  += d_found
    hybrid_hits += h_found
    rerank_hits += r_found

    print(f"{q[:48]:<50} {keyword:<28} {'✅' if d_found else '❌':>6} {'✅' if h_found else '❌':>7} {'✅' if r_found else '❌':>7}")

n = len(test_questions)
print(f"\n{'TOTAL HIT RATE':<50} {'':<28} {dense_hits/n*100:>5.0f}% {hybrid_hits/n*100:>6.0f}% {rerank_hits/n*100:>6.0f}%")
print("\n💡 Hybrid and reranking usually equal or beat dense-only on this knowledge base.")

---
## 📊 Cell 11 — Build Eval Dataset
**Slide reference:** *RAGAS & LLM-as-a-Judge*

RAGAS needs: `question`, `answer`, `contexts`, `ground_truth`

In [ ]:
from datasets import Dataset

# Ground-truth Q&A pairs (we know the correct answers)
eval_pairs = [
    {
        "question":     "How many days of annual leave do VOIS employees get?",
        "ground_truth": "VOIS employees are entitled to 21 working days of annual leave per calendar year."
    },
    {
        "question":     "What is the maximum number of remote work days per week?",
        "ground_truth": "Employees may work remotely up to 3 days per week."
    },
    {
        "question":     "What is the annual training budget for VOIS employees?",
        "ground_truth": "VOIS employees are entitled to a personal training budget of EGP 15,000 per year."
    },
    {
        "question":     "When do performance reviews take place at VOIS?",
        "ground_truth": "Performance reviews take place in June (mid-year) and December (year-end)."
    },
    {
        "question":     "What happens if an employee loses their company laptop?",
        "ground_truth": "Loss due to negligence may result in a financial deduction from the final settlement."
    },
]

# Build the dataset — simulate answers using retrieved context
# (If you have an OpenAI key, this will use GPT-4o; otherwise uses a template answer)
eval_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

for pair in eval_pairs:
    q = pair["question"]
    docs = retriever.invoke(q)
    ctx = [d.page_content for d in docs]

    if OPENAI_API_KEY.startswith("sk-") and len(OPENAI_API_KEY) > 10:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "Answer based only on the provided context. Be concise."},
                {"role": "user", "content": f"Context:\n{chr(10).join(ctx)}\n\nQuestion: {q}"},
            ],
            temperature=0, max_tokens=150,
        )
        ans = resp.choices[0].message.content
    else:
        # Fallback: use the first retrieved chunk as the "answer"
        ans = ctx[0][:300] if ctx else "No context found."

    eval_data["question"].append(q)
    eval_data["answer"].append(ans)
    eval_data["contexts"].append(ctx)
    eval_data["ground_truth"].append(pair["ground_truth"])

eval_dataset = Dataset.from_dict(eval_data)
print(f"✅ Eval dataset built: {len(eval_dataset)} rows")
print(f"\nSample row:")
print(f"  Q: {eval_dataset[0]['question']}")
print(f"  A: {eval_dataset[0]['answer'][:150]}...")

---
## 📊 Cell 12 — Run RAGAS Evaluation
**Slide reference:** *RAGAS & LLM-as-a-Judge*

RAGAS scores your pipeline on 3 metrics (0–1, higher is better):
| Metric | What it measures | Target |
|--------|-------------------|--------|
| `faithfulness` | Is answer grounded in context? | ≥ 0.85 |
| `answer_relevancy` | Does answer address the question? | ≥ 0.80 |
| `context_precision` | Are relevant chunks ranked higher? | ≥ 0.75 |

> ⚠️ RAGAS uses an LLM judge internally — needs your OpenAI key. Without it, scores will use a lightweight fallback.

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

print("⏳ Running RAGAS evaluation (may take 1–2 min)...")

try:
    results = evaluate(
        dataset=eval_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    print("\n" + "=" * 45)
    print("      RAGAS EVALUATION RESULTS")
    print("=" * 45)
    print(f"  faithfulness       : {results['faithfulness']:.3f}   (target ≥ 0.85)")
    print(f"  answer_relevancy   : {results['answer_relevancy']:.3f}   (target ≥ 0.80)")
    print(f"  context_precision  : {results['context_precision']:.3f}   (target ≥ 0.75)")
    print("=" * 45)

    # Diagnosis
    print("\n🩺 Diagnosis:")
    if results['context_precision'] < 0.75:
        print("  ❌ context_precision low → try Multi-Query or smaller chunk_size")
    else:
        print("  ✅ context_precision OK")
    if results['faithfulness'] < 0.85:
        print("  ❌ faithfulness low → try adding reranking (Cell 9)")
    else:
        print("  ✅ faithfulness OK")
    if results['answer_relevancy'] < 0.80:
        print("  ❌ answer_relevancy low → tighten your system prompt")
    else:
        print("  ✅ answer_relevancy OK")

except Exception as e:
    print(f"⚠️  RAGAS needs a valid OpenAI API key to run the LLM judge.")
    print(f"   Error: {e}")
    print("\n💡 Without an API key, here's what RAGAS would show you:")
    print("   faithfulness      ≈ 0.82–0.91  (depends on chunk quality)")
    print("   answer_relevancy  ≈ 0.78–0.88")
    print("   context_precision ≈ 0.65–0.80")

---
## 🏁 Cell 13 — Summary & Next Steps

**What you built today:**
```
Documents → Chunks (512/50) → Embeddings → ChromaDB
                                               ↓
Question → Embed → Dense Search ─┐
        → Tokenize → BM25    ─── Hybrid Fusion → Rerank → GPT-4o → Answer
                                               ↓
                                         RAGAS Scores
```

**To upgrade to Azure (production):**
```python
# Swap ChromaDB → Azure AI Search
from langchain_community.vectorstores import AzureSearch
vectorstore = AzureSearch(
    azure_search_endpoint=AZURE_SEARCH_ENDPOINT,
    azure_search_key=AZURE_SEARCH_KEY,
    index_name="vois-rag",
    embedding_function=embedding_model.embed_query,
)
# Azure AI Search has hybrid search built-in — enable with search_type='hybrid'

# Swap HuggingFace → Azure OpenAI embeddings
from langchain_openai import AzureOpenAIEmbeddings
embedding_model = AzureOpenAIEmbeddings(model="text-embedding-3-large")
```

**References:**
- RAGAS paper: https://arxiv.org/abs/2309.15217
- RAG Survey: https://arxiv.org/abs/2312.10997  
- LangChain docs: https://python.langchain.com
- MTEB Leaderboard: https://huggingface.co/spaces/mteb/leaderboard

---
*VOIS AI Team · April 2026 · Sara Zayan & Youssef Mazen*